<a href="https://colab.research.google.com/github/NBushey26/ds2002-fa26/blob/main/Copy_of_2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(tries):
        response = requests.get(BASE, params=params, timeout=15)

        if response.status_code == 200:
            return pd.DataFrame(response.json()['daily'])

        if response.status_code not in RETRY_STATUSES:
            response.raise_for_status()

        if attempt < tries - 1:
            time.sleep(attempt + 1)

    raise RuntimeError("The weather API failed after several attempts.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
used_fallback = False

try:
    weather_data = get_weather('2026-09-05', '2026-10-03')
    weather_data = weather_data[
        weather_data['time'].isin(['2026-09-05', '2026-10-03'])
    ].copy()

    if len(weather_data) != 2:
        raise ValueError("The API did not return both dates.")

except Exception as e:
    print("API request failed:", e)
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = weather_data.copy()
    wx['source'] = 'open-meteo'

print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [5]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
wx = wx.sort_values('date').reset_index(drop=True)

print(wx)

        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-10-03      22.0       14.7  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
rain_sep = wx.loc[wx['date'] == '2026-09-05', 'precip_mm'].iloc[0]
rain_oct = wx.loc[wx['date'] == '2026-10-03', 'precip_mm'].iloc[0]

difference = abs(rain_sep - rain_oct)

if rain_sep > rain_oct:
    print(f"September 5 was wetter by {difference:.1f} mm.")
elif rain_oct > rain_sep:
    print(f"October 3 was wetter by {difference:.1f} mm.")
else:
    print("Both dates had the same amount of rainfall.")

October 3 was wetter by 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
from datetime import datetime

print("Source: Open-Meteo Historical Weather API")
print("Endpoint:", BASE)
print("Coordinates: 38.03 latitude, -78.51 longitude")
print("Date range: 2026-09-05 to 2026-10-03")
print("Retrieved:", datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z'))
print("Data used:", wx['source'].iloc[0])

Source: Open-Meteo Historical Weather API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03 latitude, -78.51 longitude
Date range: 2026-09-05 to 2026-10-03
Retrieved: 2026-10-12 03:21:52 UTC
Data used: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_The merge would still be able to run, it would just have missing weather values. This would cause the chart to have misleading data, either not displaying it or at the very least being incapable of showing the relaitonship between rainfall and revenue. This could cause the rest of the data collected to be skewed and incorrect conclusions to be drawn._

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'

orlando_weather = get_weather(
    '2026-09-05', '2026-10-03',
    lat=28.54, lon=-81.38
)

orlando_weather = orlando_weather[
    orlando_weather['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

orlando_weather = orlando_weather.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

orlando_weather['date'] = pd.to_datetime(orlando_weather['date'])
orlando_weather['source'] = 'open-meteo'
orlando_weather['city'] = 'Orlando'

both_cities = pd.concat(
    [charlottesville, orlando_weather],
    ignore_index=True
)

print(both_cities)

        date  temp_max  precip_mm      source             city
0 2026-09-05      29.4        2.5  open-meteo  Charlottesville
1 2026-10-03      22.0       14.7  open-meteo  Charlottesville
2 2026-09-05      32.1        3.9  open-meteo          Orlando
3 2026-10-03      30.4       14.5  open-meteo          Orlando
